# Using IfcOpenShell in Python

A short introduction to reading and inspecting IFC model data with Python.

## 1. Setup

Install the packages with `pip install ifcopenshell pandas jupyter` and import them below.

In [ ]:
from pathlib import Path
from collections import Counter

import ifcopenshell
import ifcopenshell.util.element
import pandas as pd

print('IfcOpenShell version:', ifcopenshell.version)

## 2. Open an IFC file

Set the path to your own IFC file.

In [ ]:
IFC_PATH = Path('example.ifc')  # Change to your own model path

if not IFC_PATH.is_file():
    raise FileNotFoundError(
        f'Could not find {IFC_PATH.resolve()}. Update IFC_PATH to an existing .ifc file.'
    )

model = ifcopenshell.open(str(IFC_PATH))
print('IFC schema:', model.schema)
print('Total entities:', len(list(model)))

## 3. Find and count elements

`by_type()` retrieves elements of an IFC class.

In [ ]:
classes = ['IfcWall', 'IfcWindow', 'IfcDoor', 'IfcSpace', 'IfcBuildingStorey']
class_counts = pd.DataFrame({
    'IFC class': classes,
    'Count': [len(model.by_type(cls)) for cls in classes]
})
display(class_counts)

## 4. Read properties

Direct attributes include `Name` and `GlobalId`. Additional values may be in property sets.

In [ ]:
ELEMENT_CLASS = 'IfcWall'  # Try IfcWindow, IfcDoor or another populated class
elements = model.by_type(ELEMENT_CLASS)

if not elements:
    print(f'No {ELEMENT_CLASS} elements found. Choose another populated class.')
else:
    element = elements[0]
    print('Class:', element.is_a())
    print('Name:', element.Name)
    print('GlobalId:', element.GlobalId)
    psets = ifcopenshell.util.element.get_psets(element)
    print('Available property / quantity sets:', list(psets))
    for name, properties in list(psets.items())[:3]:
        print(f'\n{name}:', properties)

## 5. Find element locations

`get_container()` returns an element’s assigned spatial container, when available.

In [ ]:
rows = []
for element in elements:
    container = ifcopenshell.util.element.get_container(element)
    rows.append({
        "GlobalId": element.GlobalId,
        "Name": element.Name,
        "IFC class": element.is_a(),
        "Container": container.Name if container else None,
    })

element_df = pd.DataFrame(rows, columns=["GlobalId", "Name", "IFC class", "Container"])
display(element_df.head(10))


## 6. Export a table

Save the element table as a CSV file.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
element_df.to_csv(output_dir / "elements.csv", index=False)
print("Saved:", output_dir / "elements.csv")
